In [1]:
import json
from glob import glob
import numpy as np
from utils import *
import os
from tqdm import tqdm
from scipy import stats as scipy_stats

os.makedirs("Outputs", exist_ok=True)
os.makedirs("Outputs/Auto", exist_ok=True)
os.environ["CUDA_VISIBLE_DEVICES"] = "6"

/home/sahajps/US_CNs/env_CNsUS/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
human_file = "../Experiments/Postprocessed Outputs/human.json"
files = glob("../Experiments/Postprocessed Outputs/*.json")
files.sort()

dic_h = json.load(open(human_file))

for f in tqdm(files):
    dic_ai = json.load(open(f))
    scores_dic = return_auto_scores( list(dic_h.values()), list(dic_ai.values()))

    json.dump(scores_dic, open(f"Outputs/Auto/{f.split('/')[-1]}", "w"), indent=4)

  0%|          | 0/15 [00:00<?, ?it/s]Device set to use cuda:0
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
  7%|▋         | 1/15 [00:20<04:45, 20.37s/it]Device set to use cuda:0
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
 13%|█▎        | 2/15 [00:42<04:35, 21.23s/it]Device set to use cuda:0
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use i

In [3]:
def return_mean_scores(scores):
    try:
        return np.round(stats.mean(scores), 3)
    except:
        return scores

scores = {}
for f in sorted(glob("Outputs/Auto/*.json")):
    if "abl" in f:
        continue

    scores_dic = json.load(open(f))
    scores_dic = {i: return_mean_scores(scores_dic[i]) for i in scores_dic}
    scores_dic["method"] = os.path.basename(f).replace(".json", "")
    scores[scores_dic["method"]] = scores_dic

sorted_scores = []
for m in ['human', 'supernote_lite_Llama-3.1-8B-Instruct', 'supernote_lite_Ministral-8B-Instruct-2410', 'supernote_lite_Qwen3-14B', 'supernote_lite_Apriel-Nemotron-15b-Thinker', 'supernote_lite_gpt-5-nano', 'supernote_lite_gemini-2.5-flash', 'web_search_gpt-5-nano_T1', 'web_search_gpt-5-nano_T2', 'web_search_gemini-2.5-flash_T1', 'web_search_gemini-2.5-flash_T2', 'web_search_grok-4_T1', 'web_search_sonar-deep-research_T1', 'our_gpt-5-nano_T2', 'our_gemini-2.5-flash_T2']:
    sorted_scores.append(scores[m])

df = pd.DataFrame(sorted_scores)
df = df[['method', 'rouge_l', 'bert_score', 'language_bias', 'domain_bias', 'is_lowcred', 'url_recall', 'num_urls', 'note_length', 'num_of_notes']]
df.to_csv("Outputs/auto_scores.csv", index=False)

df

,method,rouge_l,bert_score,language_bias,domain_bias,is_lowcred,url_recall,num_urls,note_length,num_of_notes
0,human,1.000,1.000,0.234,-0.148,0.040,1.000,1.668,172.186,488
1,supernote_lite_Llama-3.1-8B-Instruct,0.193,0.864,0.164,-0.171,0.065,0.120,1.808,225.125,265
2,supernote_lite_Ministral-8B-Instruct-2410,0.208,0.861,0.226,-0.154,0.046,0.125,1.800,153.921,265
3,supernote_lite_Qwen3-14B,0.202,0.868,0.225,-0.135,0.044,0.122,1.525,147.208,265
4,supernote_lite_Apriel-Nemotron-15b-Thinker,0.192,0.862,0.208,-0.121,0.046,0.107,1.664,226.857,265
5,supernote_lite_gpt-5-nano,0.170,0.856,0.221,-0.115,0.041,0.129,1.823,212.219,265
6,supernote_lite_gemini-2.5-flash,0.224,0.871,0.194,-0.150,0.038,0.125,1.332,161.121,265
7,web_search_gpt-5-nano_T1,0.147,0.849,0.182,-0.223,0.066,0.040,2.920,241.572,488
8,web_search_gpt-5-nano_T2,0.145,0.849,0.183,-0.218,0.073,0.039,2.846,241.656,488
9,web_search_gemini-2.5-flash_T1,0.171,0.857,0.110,-0.182,0.067,0.043,3.213,254.701,488


In [4]:
def mean_difference_significance(list1, list2):
    x, y = np.array(list1), np.array(list2)

    _, p_value = scipy_stats.ttest_ind(x, y, equal_var=False)

    if p_value < 0.001:
        sig = "***"   # p < 0.001
    elif p_value < 0.01:
        sig = "**"    # p < 0.01
    elif p_value < 0.05:
        sig = "*"     # p < 0.05
    else:
        sig = "ns"    # not significant

    return p_value, sig

sig_test_pairs = [('web_search_gpt-5-nano_T1', 'web_search_gpt-5-nano_T2'),
                  ('supernote_lite_gpt-5-nano', 'web_search_gpt-5-nano_T2'),
                  ('web_search_gemini-2.5-flash_T1', 'web_search_gemini-2.5-flash_T2'),
                  ('supernote_lite_gemini-2.5-flash', 'web_search_gemini-2.5-flash_T2'),
                  ('web_search_gpt-5-nano_T2', 'our_gpt-5-nano_T2'),
                  ('supernote_lite_gpt-5-nano', 'our_gpt-5-nano_T2'),
                  ('web_search_gemini-2.5-flash_T2', 'our_gemini-2.5-flash_T2'),
                  ('supernote_lite_gemini-2.5-flash', 'our_gemini-2.5-flash_T2')]

for metric in ['rouge_l', 'bert_score', 'language_bias', 'domain_bias', 'is_lowcred', 'url_recall', 'num_urls', 'note_length']:
    print(f"\nSignificance tests for {metric}:\n")
    for (m1, m2) in sig_test_pairs:
        list1 = json.load(open(f"Outputs/Auto/{m1}.json"))[metric]
        list2 = json.load(open(f"Outputs/Auto/{m2}.json"))[metric]
        p_value, sig = mean_difference_significance(list1, list2)
        print(f"{m1} vs {m2}: p-value = {p_value:.4f}, significance = {sig}")


Significance tests for rouge_l:

web_search_gpt-5-nano_T1 vs web_search_gpt-5-nano_T2: p-value = 0.7272, significance = ns
supernote_lite_gpt-5-nano vs web_search_gpt-5-nano_T2: p-value = 0.0001, significance = ***
web_search_gemini-2.5-flash_T1 vs web_search_gemini-2.5-flash_T2: p-value = 0.8675, significance = ns
supernote_lite_gemini-2.5-flash vs web_search_gemini-2.5-flash_T2: p-value = 0.0000, significance = ***
web_search_gpt-5-nano_T2 vs our_gpt-5-nano_T2: p-value = 0.9222, significance = ns
supernote_lite_gpt-5-nano vs our_gpt-5-nano_T2: p-value = 0.0000, significance = ***
web_search_gemini-2.5-flash_T2 vs our_gemini-2.5-flash_T2: p-value = 0.1809, significance = ns
supernote_lite_gemini-2.5-flash vs our_gemini-2.5-flash_T2: p-value = 0.0000, significance = ***

Significance tests for bert_score:

web_search_gpt-5-nano_T1 vs web_search_gpt-5-nano_T2: p-value = 0.9093, significance = ns
supernote_lite_gpt-5-nano vs web_search_gpt-5-nano_T2: p-value = 0.0000, significance = ***